# Fourth Down Decision Tool

**Goal:** For every 4th down, estimate the expected points of each option (go for it, kick a field goal, punt) and recommend the best one.

**How it works:** the three decision models predict what happens, and the expected points (EP) model puts a value on where each outcome leaves the game.

```
GO    = P(convert) x EP(us, 1st & 10 at ytg - distance)  +  (1 - P(convert)) x -EP(them, 1st & 10 at 100 - ytg)
FG    = P(make) x 3                                       +  (1 - P(make))    x -EP(them, 1st & 10 at min(100 - ytg, 80))
PUNT  = -EP(them, 1st & 10 at predicted opp_start_ytg)
```

EP is from the view of the team with the ball, so the opponent's EP counts against us (the minus signs). A missed field goal goes back to the previous spot, or the 20 if the kick was from inside the 20.

**Models** (saved by the last cell of each notebook in this folder): `ep_model.joblib`, `go_model.keras` + `go_prep.joblib`, `fg_model.joblib`, `punt_model.joblib`

In [2]:
from cfb.ml import *

models_dir = ROOT / 'models'

ep       = joblib.load(models_dir / 'ep_model.joblib')     # {'model', 'features', 'points'}
fg       = joblib.load(models_dir / 'fg_model.joblib')     # {'model', 'features', 'league_rate'}
punt     = joblib.load(models_dir / 'punt_model.joblib')   # {'model', 'features'}
go_prep  = joblib.load(models_dir / 'go_prep.joblib')      # {'preprocessor', 'features'}
go_model = tf.keras.models.load_model(models_dir / 'go_model.keras')

## Data

`sql/analysis/fourth_down_decisions.sql` returns every 4th down in a season, whatever the coach chose, with the columns all four models need. The go-for-it history and kicker history are rebuilt for every play, not just the plays where that choice was made.

In [3]:
SEASON = 2025   # the test season: none of the models were tuned on it

plays = read_query('fourth_down_decisions.sql', season=SEASON)
plays = plays[~plays['is_overtime']].reset_index(drop=True)      # none of the models handle overtime

# The training notebooks fed true/false columns to the models as 1.0 / 0.0
bool_cols = plays.select_dtypes('bool').columns
plays[bool_cols] = plays[bool_cols].astype(float)

print(plays.shape)
plays['decision'].value_counts()

(14372, 47)


decision
punt          7859
go            3873
field goal    2640
Name: count, dtype: int64

## Expected points of a game state

Every option ends in "someone has 1st & 10 somewhere", so one helper values all of them. It copies the game context (season, half, clock) from the real play and fills in the new field position. To value a state for the opponent, pass their yards to goal (`100 - ytg`) and their score margin (`-score_margin`), then flip the sign of the result.

In [4]:
ep_points = np.array([ep['points'][c] for c in ep['model'].classes_])   # points in the model's class order


def ep_value(plays, yards_to_goal, score_margin):
    """EP for whoever has the ball, 1st & 10 at yards_to_goal."""
    ytg = np.clip(np.asarray(yards_to_goal, dtype=float), 1, 99)
    states = pd.DataFrame({
        'season'           : plays['season'].to_numpy(),
        'half'             : np.where(plays['period'] <= 2, 1, 2),
        'sec_left_in_half' : plays['sec_left_in_half'].to_numpy(),
        'down'             : 1,
        'distance'         : np.minimum(10, ytg),            # 1st & 10, or 1st & goal inside the 10
        'yards_to_goal'    : ytg,
        'is_goal_to_go'    : (ytg <= 10).astype(float),
        'score_margin'     : np.asarray(score_margin, dtype=float),
    })[ep['features']]                                       # the column order the model was trained on
    return ep['model'].predict_proba(states) @ ep_points

## The three options

In [5]:
def go_value(plays):
    """Expected points of going for it, and the conversion probability."""
    X = go_prep['preprocessor'].transform(plays[go_prep['features']]).astype('float32')
    p = go_model.predict(X, verbose=0).ravel()

    new_ytg = plays['yards_to_goal'] - plays['distance']
    success = np.where(new_ytg <= 0, 7,                                             # converting on goal-to-go is a touchdown
                       ep_value(plays, new_ytg, plays['score_margin']))
    fail = -ep_value(plays, 100 - plays['yards_to_goal'], -plays['score_margin'])   # turnover on downs at the spot
    return p * success + (1 - p) * fail, p


def fg_value(plays, max_kick=65):
    """Expected points of kicking, and the make probability."""
    # rebuild the shrunk kicking rate exactly as the field goal notebook did
    rate = fg['league_rate']
    attempts = plays['fg_att_szn'].fillna(0)
    makes = (plays['fg_pct_szn'] * plays['fg_att_szn']).fillna(0)
    X = plays.assign(
        fg_shrunk_rate = (makes + 10 * rate) / (attempts + 10),
        opp_start_ytg  = np.nan,                 # punt-only columns: always empty on field goal plays
        net_punt_yards = np.nan,
    )[fg['features']]
    q = fg['model'].predict_proba(X)[:, 1]

    miss_ytg = np.minimum(100 - plays['yards_to_goal'], 80)
    value = q * 3 + (1 - q) * -ep_value(plays, miss_ytg, -plays['score_margin'])
    value = np.where(plays['kick_distance_fixed'] > max_kick, -np.inf, value)       # out of range: not an option
    return value, q


def punt_value(plays):
    """Expected points of punting, and the predicted opponent start."""
    opp_start = punt['model'].predict(plays[punt['features']]).clip(1, 99)
    return -ep_value(plays, opp_start, -plays['score_margin']), opp_start

## Combine

In [6]:
DECISION_NAMES = {'go': 'go', 'field goal': 'fg', 'punt': 'punt'}   # data labels -> recommendation labels


def fourth_down_decision(plays):
    go, p_convert = go_value(plays)
    fgv, p_make = fg_value(plays)
    pt, opp_start = punt_value(plays)

    out = plays[['game_id', 'play_id', 'week', 'offense', 'defense', 'period', 'decision',
                 'distance', 'yards_to_goal', 'score_margin']].copy()
    out['p_convert'], out['p_make'], out['pred_opp_start'] = p_convert, p_make, opp_start
    out['ev_go'], out['ev_fg'], out['ev_punt'] = go, fgv, pt

    ev = out[['ev_go', 'ev_fg', 'ev_punt']]
    out['recommend'] = ev.idxmax(axis=1).str.removeprefix('ev_')
    top_two = np.sort(ev.to_numpy(), axis=1)[:, -2:]
    out['margin'] = top_two[:, 1] - top_two[:, 0]              # how clear the call is, in points

    out['coach'] = out['decision'].map(DECISION_NAMES)
    chosen = out['coach'].map({'go': 0, 'fg': 1, 'punt': 2}).to_numpy()
    out['ev_coach'] = ev.to_numpy()[np.arange(len(out)), chosen]
    out['ep_lost'] = ev.max(axis=1) - out['ev_coach']           # 0 when the coach agreed with the model
    return out

## Sanity check: spots with an obvious answer

Same game situation (taken from a real 1st-quarter play with a tied score), different down and distance. If these don't come out as expected, look at `p_convert`, `p_make` and `pred_opp_start` to see which model is off.

In [7]:
base = plays[(plays['period'] == 1) & (plays['score_margin'] == 0)].iloc[[0]]

def scenario(ytg, distance):
    row = base.copy()
    row['yards_to_goal'] = ytg
    row['distance'] = distance
    row['kick_distance'] = row['kick_distance_fixed'] = ytg + 17
    row['is_goal_to_go'] = float(distance >= ytg)
    return row

spots = {
    "4th & 1 at midfield (go)"        : scenario(50, 1),
    "4th & 10 at own 25 (punt)"       : scenario(75, 10),
    "4th & 6 at opp 15 (fg)"          : scenario(15, 6),
    "4th & goal at the 1 (go)"        : scenario(1, 1),
    "4th & 4 at opp 38 (close call)"  : scenario(38, 4),
}
checks = fourth_down_decision(pd.concat(spots.values(), ignore_index=True))
checks.index = spots.keys()
checks[['p_convert', 'p_make', 'pred_opp_start', 'ev_go', 'ev_fg', 'ev_punt', 'recommend', 'margin']].round(2)

c:\Users\Owner\projects\cfb-project\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['net_punt_yards']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(


,p_convert,p_make,pred_opp_start,ev_go,ev_fg,ev_punt,recommend,margin
4th & 1 at midfield (go),0.68,0.47,85.160004,1.02,-inf,-0.37,go,1.39
4th & 10 at own 25 (punt),0.35,0.67,65.580002,-2.20,-inf,-1.72,punt,0.48
4th & 6 at opp 15 (fg),0.46,0.85,48.450001,2.29,2.43,-2.87,fg,0.14
4th & goal at the 1 (go),0.58,0.96,49.310001,4.45,2.85,-2.78,go,1.60
4th & 4 at opp 38 (close call),0.51,0.45,87.190002,0.96,0.25,-0.32,go,0.71


## Every 4th down in the season

In [8]:
results = fourth_down_decision(plays)

agree = (results['coach'] == results['recommend']).mean()
print(f"Coaches agreed with the model on {agree:.1%} of {len(results):,} fourth downs")
print(f"Expected points left on the table: {results['ep_lost'].sum():,.0f} total, "
      f"{results['ep_lost'].mean():.3f} per fourth down")

# rows: what the coach did, columns: what the model recommends
pd.crosstab(results['coach'], results['recommend'], margins=True)

c:\Users\Owner\projects\cfb-project\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['net_punt_yards']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(


Coaches agreed with the model on 61.3% of 14,372 fourth downs
Expected points left on the table: 3,594 total, 0.250 per fourth down


recommend,fg,go,punt,All
coach,,,,
fg,1912,720,8,2640
go,826,2774,273,3873
punt,510,3229,4120,7859
All,3248,6723,4401,14372


### Clearest disagreements

Plays where the coach's choice cost the most expected points. Check a few against the play-by-play: a big `ep_lost` with an odd `p_convert` or `p_make` points at a model problem, not a coaching one.

In [9]:
results.sort_values('ep_lost', ascending=False).head(15)[
    ['week', 'offense', 'defense', 'period', 'distance', 'yards_to_goal', 'score_margin',
     'coach', 'recommend', 'p_convert', 'p_make', 'ev_go', 'ev_fg', 'ev_punt', 'ep_lost']
].round(2)

,week,offense,defense,period,distance,yards_to_goal,score_margin,coach,recommend,p_convert,p_make,ev_go,ev_fg,ev_punt,ep_lost
4774,4,Duquesne,Akron,3,4,4,-38,go,fg,0.27,0.94,-1.41,2.52,-5.56,3.93
9191,10,Sam Houston,Louisiana Tech,4,18,23,-27,go,fg,0.10,0.70,-2.05,1.27,-3.87,3.32
9301,10,Vanderbilt,Texas,4,19,98,-10,go,punt,0.18,0.83,-3.99,-inf,-0.76,3.22
4026,4,Auburn,Oklahoma,4,29,96,-5,go,punt,0.05,0.71,-3.21,-inf,-0.23,2.98
3270,3,Texas Tech,Oregon State,2,65,93,14,go,punt,0.00,0.72,-4.39,-inf,-1.42,2.97
11772,13,Coastal Carolina,South Carolina,3,4,17,-37,go,fg,0.39,0.81,-1.35,1.55,-5.32,2.91
7596,8,South Carolina,Oklahoma,4,36,72,-17,go,punt,0.01,0.58,-3.45,-inf,-0.62,2.82
9306,10,Maryland,Indiana,3,5,5,-31,go,fg,0.30,0.95,-0.09,2.69,-4.64,2.78
6868,7,Oregon State,Wake Forest,3,5,17,-32,go,fg,0.29,0.82,-0.90,1.87,-4.56,2.77
7704,8,Michigan State,Indiana,4,13,23,-25,go,fg,0.18,0.74,-1.14,1.57,-3.84,2.71
